# Opt-in tags

> **Install for this page:** `pip install scope-profiler` (base install only).

A library can ship its own regions without costing its users anything: it
registers an *opt-in tag*, tags its regions with it, and those regions stay off
(a cheap pass-through call) until a run enables the tag. This page builds a
small instrumented library, lists its tags with `scope-profiler tags` and turns
them on from the command line, the environment and Python.

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

# Work in a scratch directory so nothing lands in the working directory,
# and make the `scope-profiler` command of this environment available.
WORKDIR = Path(tempfile.mkdtemp(prefix="scope-profiler-guide-"))
os.chdir(WORKDIR)
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]
os.environ["PYTHONPATH"] = os.pathsep.join(
    filter(None, [str(WORKDIR), os.environ.get("PYTHONPATH")])
)
sys.path.insert(0, str(WORKDIR))

## An instrumented library

Name the regions `<tag>.<operation>` and register the tag once, at import time.
Keep the tags in one place so the decorators and the registration agree.

In [ ]:
%%writefile mylib.py
import time

import scope_profiler as sp

PROFILE_TAGS = ("mylib",)
sp.register_opt_in_tag(*PROFILE_TAGS)


@sp.profile("mylib.solve", tags=PROFILE_TAGS)
def solve():
    time.sleep(0.002)


@sp.profile("mylib.write", tags=PROFILE_TAGS)
def write():
    time.sleep(0.001)

In [ ]:
%%writefile app.py
import scope_profiler as sp

import mylib

for _ in range(3):
    with sp.region("step"):
        mylib.solve()
        mylib.write()

## Listing the tags

`scope-profiler tags` imports the packages that declare a
`scope_profiler.opt_in_tags` entry point and prints each tag with the regions
behind it. `--import` adds a module that has no entry point, and `--json` gives
the same as a JSON document.

In [ ]:
%%bash
scope-profiler tags --no-entry-points --import mylib
scope-profiler tags --no-entry-points --import mylib --json

A package makes its tags discoverable with an entry point in its
`pyproject.toml`; the value is the module that registers them:

```toml
[project.entry-points."scope_profiler.opt_in_tags"]
mylib = "mylib"
```

In Python, `sp.opt_in_tags()` returns the same mapping.

In [ ]:
import mylib

import scope_profiler as sp

sp.opt_in_tags()

## Turning the tags on

Without a tag, only the script's own regions are recorded. `--enable-tag`
(repeatable; `'*'` enables every opt-in tag) turns the library's regions on.

In [ ]:
%%bash
scope-profiler run -q -o off.h5 app.py
scope-profiler run -q -o on.h5 --enable-tag mylib app.py
scope-profiler inspect off.h5 --regions-only
scope-profiler inspect on.h5 --regions-only

The environment variable `SCOPE_PROFILER_ENABLE_TAGS` takes a comma-separated
list and adds to the other ways, which helps when the run is started by
another tool:

In [ ]:
%%bash
SCOPE_PROFILER_ENABLE_TAGS=mylib scope-profiler run -q -o env.h5 app.py
scope-profiler inspect env.h5 --regions-only

In code, pass `enable_tags` to `sp.setup`, `sp.session` or
`ProfilingOptions`:

In [ ]:
with sp.session(output="session.h5", enable_tags=["mylib"], verbose=False):
    mylib.solve()

sorted(region.name for region in sp.read_h5("session.h5").get_regions())

Keep library regions coarse (communication, I/O, solves, whole kernels): a
disabled region still costs a function call, so tagging a tight inner loop is
not free. See [Overhead](/scope-profiler/guide/overhead/) for the numbers.